# Data Cleaning and Preprocessing

This notebook takes the raw scraped data, cleans the `price` and `availability` columns into usable numerical formats, and checks for missing or corrupted data.

In [ ]:
import pandas as pd
import numpy as np
import os

data_path = 'books_data_with_images.csv'

if not os.path.exists(data_path):
    print(f"File not found: {data_path}.")
else:
    df = pd.read_csv(data_path)
    display(df.head())

,title,category,image_url,price,star_rating,availability,local_image_path
0,Soumission,Fiction,http://books.toscrape.com/media/cache/3e/ef/3e...,Â£50.10,1,In stock,data/images/book_0000.jpg
1,Private Paris (Private #10),Fiction,http://books.toscrape.com/media/cache/9d/05/9d...,Â£47.61,5,In stock,data/images/book_0001.jpg
2,"We Love You, Charlie Freeman",Fiction,http://books.toscrape.com/media/cache/5f/15/5f...,Â£50.27,5,In stock,data/images/book_0002.jpg
3,Thirst,Fiction,http://books.toscrape.com/media/cache/c4/0a/c4...,Â£17.27,5,In stock,data/images/book_0003.jpg
4,The Murder That Never Was (Forensic Instincts #5),Fiction,http://books.toscrape.com/media/cache/dc/44/dc...,Â£54.11,3,In stock,data/images/book_0004.jpg


## 1. Clean the Price Column
Currently, `price` is a string like `£22.50` or `Â£22.50`. We will extract the numeric value and convert it to a float.

In [ ]:
df['price'] = df['price'].astype(str).str.extract(r'(\d+\.\d+)').astype(float)

print("Cleaned Price:")
display(df[['title', 'price']].head())

Cleaned Price:


,title,price
0,Soumission,50.10
1,Private Paris (Private #10),47.61
2,"We Love You, Charlie Freeman",50.27
3,Thirst,17.27
4,The Murder That Never Was (Forensic Instincts #5),54.11


## 2. Clean the Availability Column
The `availability` column looks like `In stock (19 available)`. We will extract the integer number of available copies.

In [ ]:
# We will fill NaNs with 0 in case there are strings like 'Out of stock' without numbers
df['availability_count'] = df['availability'].astype(str).str.extract(r'(\d+)').fillna(0).astype(int)

df['in_stock'] = df['availability_count'] > 0

print("Cleaned Availability:")
display(df[['title', 'availability', 'availability_count', 'in_stock']].head())

Cleaned Availability:


,title,availability,availability_count,in_stock
0,Soumission,In stock,0,False
1,Private Paris (Private #10),In stock,0,False
2,"We Love You, Charlie Freeman",In stock,0,False
3,Thirst,In stock,0,False
4,The Murder That Never Was (Forensic Instincts #5),In stock,0,False


## 3. Verify Star Ratings
The star ratings should be integers (1 to 5). We will verify their distribution.

In [4]:
print("Value counts for star ratings:")
print(df['star_rating'].value_counts().sort_index())

Value counts for star ratings:
star_rating
1    37
2    29
3    44
4    28
5    37
Name: count, dtype: int64


## 4. Check for Corrupted or Missing Data
We will check if any rows have missing titles, categories, or failed image downloads.

In [5]:
# Check for general missing values
print("Missing values per column:")
print(df.isnull().sum())

# Find rows where image download failed
missing_images = df[df['local_image_path'].isnull()]
if len(missing_images) > 0:
    print(f"\nWarning: {len(missing_images)} books failed to download their images.")
    display(missing_images)
else:
    print("\nAll images were successfully downloaded and linked!")

# Drop rows with missing essential data if needed
df = df.dropna(subset=['local_image_path', 'title', 'category'])

Missing values per column:
title                 0
category              0
image_url             0
price                 0
star_rating           0
availability          0
local_image_path      0
availability_count    0
in_stock              0
dtype: int64

All images were successfully downloaded and linked!


## 5. Save Cleaned Dataset
We will save this clean data to a new CSV file.

In [ ]:
# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

cleaned_csv_path = 'data/cleaned_books_data.csv'
df.to_csv(cleaned_csv_path, index=False)
print(f"Cleaned dataset saved to {cleaned_csv_path}")

df.info()

Cleaned dataset saved to data/cleaned_books_data.csv
<class 'pandas.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               175 non-null    str    
 1   category            175 non-null    str    
 2   image_url           175 non-null    str    
 3   price               175 non-null    float64
 4   star_rating         175 non-null    int64  
 5   availability        175 non-null    str    
 6   local_image_path    175 non-null    str    
 7   availability_count  175 non-null    int64  
 8   in_stock            175 non-null    bool   
dtypes: bool(1), float64(1), int64(2), str(5)
memory usage: 11.2 KB
